# Geração das instâncias NRP-S

Gera as instâncias com `generate_nrps` (o mesmo código do comando `salbpgen nrps`) e grava, em `instances/`, um `.nrp` (formato lido por `nrp_instance.py`) e um `.json` com os parâmetros e o caminho crítico de cada instância. O `viewer.py` lê essa pasta.

**Unidade:** T e as durações `h` estão em horas. Cada tarefa recebe `h = ⌈SP × HOURS_PER_SP⌉`, com o SP sorteado da distribuição do TAWOS.

Os parâmetros e as fontes de cada valor estão em `docs/geracao-instancias.md`. **Os níveis abaixo são exemplos**: |R|, T, ρ, α e o número de features críticas ainda estão pendentes (seção 6 do documento).

## Parâmetros

In [ ]:
from itertools import product
from pathlib import Path

import pandas as pd

from salbpgen import FeatureParams, critical_path, generate_nrps

OUT = Path("instances")

SIZES = [150, 600, 1500, 3000]   # |R|, na faixa das clássicas nrp1–nrp5 (140–3.250)
HORIZONS = [40, 80, 120, 160]    # T, os mesmos dos experimentos com as clássicas
N_CRITICAL = [2]                 # features com caminho crítico > T
CP_RATIOS = [1.2]                # ρ = CP da feature crítica / T (dev mais rápido)
ALPHAS = [1.0]                   # valor por profundidade: 0 uniforme, alto = fim da feature
SEEDS = [1]                      # réplicas

# Conversão SP → períodos de T. T está em HORAS, então HOURS_PER_SP = horas por story point.
# 8 = convenção do SAFe (1 SP ≈ 1 dia-pessoa); não é valor medido. Ver docs/geracao-instancias.md.
HOURS_PER_SP = 8

# Demais parâmetros: padrões de FeatureParams (calibrados pelo TAWOS)
EXTRA = dict(speedup=1.4, critical_width=2, feature_os=0.1, inter_share=0.35, periods_per_sp=HOURS_PER_SP)

## Geração

Uma combinação que não cabe (as features críticas precisam de mais tarefas que |R|) é pulada e aparece na tabela com o erro.

In [ ]:
OUT.mkdir(exist_ok=True)
rows = []
for n, T, k, rho, alpha, seed in product(SIZES, HORIZONS, N_CRITICAL, CP_RATIOS, ALPHAS, SEEDS):
    name = f"nrps_n{n}_T{T}_k{k}_cp{rho}_a{alpha}_h{HOURS_PER_SP}_s{seed}"
    row = dict(instancia=name, n=n, T=T, h_por_sp=HOURS_PER_SP, criticas=k, cp_ratio=rho, alpha=alpha, seed=seed)
    try:
        params = FeatureParams(n_tasks=n, horizon=T, n_critical=k, cp_ratio=rho, alpha=alpha, **EXTRA)
        inst = generate_nrps(params, seed=seed)
    except (ValueError, RuntimeError) as e:
        rows.append({**row, "erro": str(e)})
        continue
    inst.write(OUT / f"{name}.nrp")
    cp = critical_path(inst, params.speedup)
    rows.append({**row, "features": len(inst.features), "arcos": len(inst.arcs),
                 "CP_controladas": [f.critical_path_fast for f in inst.features if f.controlled],
                 "features_criticas (CP>T)": sum(f.critical for f in inst.features),
                 "CP_instancia": cp, "CP/T": round(cp / T, 2),
                 "% tarefas com h > T (MID)": round(100 * sum(h > T for h in inst.durations) / inst.n, 1),
                 "erro": None})

df = pd.DataFrame(rows)
df

## Resumo

In [ ]:
ok = df[df["erro"].isna()]
print(f"{len(ok)} instâncias geradas em {OUT.resolve()}, {len(df) - len(ok)} combinações puladas")
df[df["erro"].notna()][["instancia", "erro"]]

Para visualizar:

```bash
streamlit run viewer.py -- instances
```